In [63]:
import random
discount = 0.5

def step_left(state):
  if(state!=1):
    return state-1
  else:
    return -1

def step_right(state):
  if(state!=6):
    return state+1
  else:
    return -1

def step_null(state):
  return state

# print("\n\n")
# for i in range(1,7):
#   print(f" {step_left(i)} <- state {i} ")
#   print(f"state {i} -> {step_right(i)} ")
# print("\n\n")

def new_state(state,action):
  if(action == "L"):
    return step_left(state)
  elif(action == "R"):
    return step_right(state)
  else:
    return step_null(state)

def reward(state):
  if(state==1):
    return 100
  elif(state==6):
    return 40
  else:
    return 0

#all right policy, absolute, returns probability for L and R
def policy_r(s,a):
  if(s!=1 and s!=6):
    if(a=="R"):
      return 1
    else:
      return 0
  elif(a=="0"):
    return 1
  else:
    return 0

def policy_l(s,a):
  if(s!=1 and s!=6):
    if(a=="R" or a=="0"):
      return 0
    else:
      return 1
  elif(a=="0"):
    return 1
  else:
    return 0

def policy_mixed(s,a):
  if(s==6 or s==1):
    return 1 if(a=="0") else 0
  elif(s==5):
    return 1 if(a=="R") else 0
  else:
    return 1 if(a=="L") else 0

def policy_weighted(s,a):
  if(s==6 or s==1):
    return 1 if(a=="0") else 0
  else:
    return 0.6 if (a=="L") else 0.4 if(a=="R") else 0.0


def policy_random(s,a):
  if(s==6 or s==1):
    return 1 if(a=="0") else 0
  else:
    return 1/2 if (a=="L") else 1/2 if(a=="R") else 0

def policy_random2(s,a):
  if(s==6 or s==1):
    return 1 if(a=="0") else 0
  else:
    return 1/3 if (a=="L") else 1/3 if(a=="R") else 1/3




def sampler(state, policy_dist):
  trajectory = []
  s_init = state
  count = 0
  while(s_init!=1 and s_init!=6):

    action = random.choices(["L","0","R"],weights=policy_dist(s_init),k=1)[0]

    #print(f"current state: {s_init} action:{action} new state: {new_state(s_init,action)} count: {count} policy dist: {policy_dist(s_init)}")
    #print(f"{[s_init,action]}")
    trajectory.append([s_init,action,0,0])
    s_init = new_state(s_init,action)
    count = count + 1
    if((s_init!=6 or s_init!=1) and count>20000):
      return None
      break
  trajectory.append([s_init,random.choices(["L","0","R"],weights=policy_dist(s_init),k=1)[0],0,0])
  return trajectory


def calculate_return(trajectory):

  for i in range(len(trajectory) - 1, -1, -1):
    if(trajectory[i][0]==6 or trajectory[i][0]==1):
      trajectory[i][2] = reward(trajectory[i][0])
      trajectory[i][3] = trajectory[i][2]
    else:
      trajectory[i][3] = reward(trajectory[i][0]) + discount * trajectory[i+1][3]

  return trajectory

  #returned trajectory looks like [s,"action",reward,return]



def sampler(state, policy_dist,returns=None):
  trajectory = []
  s_init = state
  count = 0
  

  while(s_init!=1 and s_init!=6):

    if returns is None:
      action = random.choices(["L","0","R"],weights=policy_dist(s_init),k=1)[0]
    
    else:
      action = random.choices(["L","0","R"],weights=policy_dist(s_init,returns),k=1)[0]

    #print(f"current state: {s_init} action:{action} new state: {new_state(s_init,action)} count: {count} policy dist: {policy_dist(s_init)}")
    #print(f"{[s_init,action]}")
    trajectory.append([s_init,action,0,0])
    s_init = new_state(s_init,action)
    count = count + 1
    if((s_init!=6 or s_init!=1) and count>20000):
      return None
      break
  trajectory.append([s_init,random.choices(["L","0","R"],weights=policy_dist(s_init),k=1)[0],0,0])
  return trajectory




In [64]:

def policy_dist_default(s):

  return [policy_random2(s,"L"),policy_random2(s,"0"),policy_random2(s,"R")]

print("state:--- policy distribution:  L  0  R")
for i in range(1,7):
  print(f" {i}        policy distribution:  {policy_dist_default(i)[0]}  {policy_dist_default(i)[1]}  {policy_dist_default(i)[2]}")


trajectory = sampler(4, policy_dist_default)

trajectory = calculate_return(trajectory)
print("state   action   reward   return")
for i in trajectory:
  print(f"{i[0]}       {i [1]}        {i [2]}        {i [3]}")



state:--- policy distribution:  L  0  R
 1        policy distribution:  0  1  0
 2        policy distribution:  0.3333333333333333  0.3333333333333333  0.3333333333333333
 3        policy distribution:  0.3333333333333333  0.3333333333333333  0.3333333333333333
 4        policy distribution:  0.3333333333333333  0.3333333333333333  0.3333333333333333
 5        policy distribution:  0.3333333333333333  0.3333333333333333  0.3333333333333333
 6        policy distribution:  0  1  0
state   action   reward   return
4       R        0        10.0
5       R        0        20.0
6       0        40        40


In [65]:
returns = {}
counts = {}

#finds an average return value for each (s,a) combination
def return_estimation(start_state, num_episodes,policy_dist,rets=None):
    returns = {}
    counts = {}

    for i in range(num_episodes):
      if rets is None:
        #("we are here")
        trajectory = sampler(start_state,policy_dist)
        #(trajectory)
      else:
        trajectory = sampler(start_state,policy_dist,rets)
        
      if trajectory is not None:
        #print(trajectory)
        trajectory = calculate_return(trajectory)
        for state, action, r, Gt in trajectory:
            key = (state, action)
            if key in returns:
                counts[key] += 1
                # update running average
                returns[key] += (Gt - returns[key]) / counts[key]
            else:
                returns[key] = Gt
                counts[key] = 1

    return returns



returns = return_estimation(4, 10,policy_dist_default)
print(returns)
# Print results
print("state   action   average_return")
for (state, action), avg_return in sorted(returns.items()):
    print(f"{state:<7}{action:<8}{avg_return:.6f}")


{(4, 'L'): 6.64501953125, (3, 'L'): 14.507378472222221, (2, 'R'): 1.6276041666666667, (3, 'R'): 3.203125, (2, 'L'): 50.0, (1, '0'): 100.0, (4, 'R'): 5.295758928571429, (5, 'L'): 3.8802083333333335, (4, '0'): 3.3984375, (3, '0'): 0.142578125, (2, '0'): 14.583333333333332, (5, 'R'): 20.0, (6, '0'): 40.0, (5, '0'): 5.833333333333333}
state   action   average_return
1      0       100.000000
2      0       14.583333
2      L       50.000000
2      R       1.627604
3      0       0.142578
3      L       14.507378
3      R       3.203125
4      0       3.398438
4      L       6.645020
4      R       5.295759
5      0       5.833333
5      L       3.880208
5      R       20.000000
6      0       40.000000


In [66]:
def modified_policy(s,a,rets):
  Q1 = rets.get((s,"L"),0)
  Q2 = rets.get((s,"0"),0)
  Q3 = rets.get((s,"R"),0)
  sum = Q1 + Q2 + Q3
  if sum==0:
    return 1/3
  pa_s = (Q1 if(a=="L") else Q2 if(a=="0") else Q3)/sum
  return pa_s


def policy_dist_eval(s,rets=None):
  if rets is None:
    return policy_dist_default(s)
  return [modified_policy(s,"L",rets),modified_policy(s,"0",rets),modified_policy(s,"R",rets)]


def eval(s=4,policy_dist=policy_dist_default,rets=None):
  if rets is None:
    returns = return_estimation(4, 10,policy_dist_default)
  else:
    returns=rets

  trajectory = sampler(s,policy_dist,returns)
  trajectory = calculate_return(trajectory)
  # print("state   action   reward   return")
  # for i in trajectory:
  #   print(f"{i[0]}       {i [1]}        {i [2]}        {i [3]}")
  #print("Trajectory")
  states = [row[0] for row in trajectory]
  #print(states)
  return states



print("state:--- policy distribution:  L  0  R")
for i in range(1,7):
  print(f" {i}        policy distribution:  {policy_dist_eval(i,returns)[0]}  {policy_dist_eval(i,returns)[1]}  {policy_dist_eval(i,returns)[2]}")





state:--- policy distribution:  L  0  R
 1        policy distribution:  0.0  1.0  0.0
 2        policy distribution:  0.7551622418879056  0.22025565388397245  0.024582104228121928
 3        policy distribution:  0.8125980040599511  0.007986191303925024  0.17941580463612386
 4        policy distribution:  0.43320464022773675  0.2215522298467961  0.34524312992546713
 5        policy distribution:  0.13058720420683612  0.19631901840490798  0.6730937773882559
 6        policy distribution:  0.0  1.0  0.0


In [67]:



def naive_RL(iterations=100):
  print("state:--- policy distribution:  L  0  R BEFORE ITERATION")
  for i in range(1,7):
    print(f" {i}        policy distribution:  {policy_dist_default(i)[0]}  {policy_dist_default(i)[1]}  {policy_dist_default(i)[2]}")

  #convergence achieved at 100000
  returns = return_estimation(4, 10,policy_dist_default)
  
  for i in range(iterations):
    
    returns = return_estimation(4, 50,policy_dist_eval,returns)


  print(f"state:--- policy distribution:  L  0  R after all Iterations ")
  for i in range(1,7):
    print(f" {i}        policy distribution:  {policy_dist_eval(i,returns)[0]}  {policy_dist_eval(i,returns)[1]}  {policy_dist_eval(i,returns)[2]}")

  return returns



returns = naive_RL(50000)

#THIS IS OUR OPTIMAL POLICY, or rather OUR DISTRIBUTION

print("After dust is settled")
 
for i in range(1,7):
    print(f" {i}        policy distribution:  {policy_dist_eval(i,returns)[0]}  {policy_dist_eval(i,returns)[1]}  {policy_dist_eval(i,returns)[2]}")

  

state:--- policy distribution:  L  0  R BEFORE ITERATION
 1        policy distribution:  0  1  0
 2        policy distribution:  0.3333333333333333  0.3333333333333333  0.3333333333333333
 3        policy distribution:  0.3333333333333333  0.3333333333333333  0.3333333333333333
 4        policy distribution:  0.3333333333333333  0.3333333333333333  0.3333333333333333
 5        policy distribution:  0.3333333333333333  0.3333333333333333  0.3333333333333333
 6        policy distribution:  0  1  0
state:--- policy distribution:  L  0  R after all Iterations 
 1        policy distribution:  0.0  1.0  0.0
 2        policy distribution:  1.0  0.0  0.0
 3        policy distribution:  1.0  0.0  0.0
 4        policy distribution:  0.5555555555555556  0.0  0.4444444444444444
 5        policy distribution:  0.0  0.0  1.0
 6        policy distribution:  0.0  1.0  0.0
After dust is settled
 1        policy distribution:  0.0  1.0  0.0
 2        policy distribution:  1.0  0.0  0.0
 3        policy 

In [68]:
track = []
for i in range(100):
  track.append(eval(4,policy_dist_eval,returns))


from collections import Counter
# Convert inner lists to tuples
track_tuples = [tuple(x) for x in track]
# Count occurrences
counts = Counter(track_tuples)
# Total number of elements
total = len(track)
# Calculate percentages
percentages = {key: (value / total) * 100 for key, value in counts.items()}
# Sort percentages by value descending and take top 3
top3 = sorted(percentages.items(), key=lambda x: x[1], reverse=True)[:]
# Print nicely
for element, pct in top3:
    print(f"{list(element)}: {pct:.2f}%")  # convert back to list if needed


[4, 3, 2, 1]: 57.00%
[4, 5, 6]: 43.00%
